# Stage 0 — Hypothesis v3: Time-Series Momentum (TSMOM)

**Date locked:** 2026-05-24  
**Author:** Anthony Chung  
**Version:** v3.0  
**Predecessor:** v1.0 Asian fade (FAILED), v2.0 Stretched VP fade (FAILED)  
**Status:** ACTIVE

---

## Why v3

v1 and v2 were both **reversal** strategies (fade extremes). Both failed Stage 2 with 0% profitable trials.

[Yang (2018) Behavioral Anomalies in Cryptocurrency Markets](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3174421) tested 20+ stock anomalies on crypto and concluded:
- **Momentum WORKS** on crypto
- **Short-term reversal does NOT WORK**
- **Long-term reversal does NOT WORK**

Our v1+v2 failures empirically confirmed Yang's predictions. v3 reverses direction and tests momentum.

Primary academic reference: [Han, Kang, Ryu (2023) Time-Series and Cross-Sectional Momentum in the Cryptocurrency Market](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4675565). Also rooted in Moskowitz, Ooi, Pedersen (2012) seminal TSMOM paper across asset classes.

## Thesis

At each weekly rebalance, compute each coin's trailing N-week return. If positive (beyond threshold), go LONG that coin; if negative, go SHORT; if near zero, stay FLAT. Hold one week, then re-evaluate.

## Market Inefficiency Targeted

**Trend-continuation via behavioural under-reaction**:

1. New information enters crypto markets unevenly — initial reaction is incomplete (under-reaction phase).
2. As price extends in the direction of news, retail and momentum-following participants pile in (herding).
3. Funding-rate and leverage dynamics amplify directional flows (positive feedback).
4. The result: returns are positively autocorrelated at the 1-12 week horizon.
5. By the time the trend exhausts and reverses, the TSMOM signal has already flipped (the strategy exits).

This is the same mechanism documented in equity (since Jegadeesh & Titman 1993), commodity, and FX markets — but with faster turnover in crypto due to higher retail participation.

## Strategy Specification

### Universe
- 6 coins: BTC, ETH, SOL, AVAX, XRP, BNB (TONY 主 5 + BNB added for completeness)
- Each coin evaluated **independently** (this is what 'time-series' means — no cross-asset ranking)

### Rebalance schedule
- Every Monday 00:00 UTC
- Evaluate signal for each coin → adjust position if changed

### Signal computation (per coin)
```
trailing_return = price[now] / price[now - lookback_weeks * 168 hours] - 1

if trailing_return > +signal_threshold:
    position = LONG
elif trailing_return < -signal_threshold:
    position = SHORT
else:
    position = FLAT  # avoid noise zone
```

### Execution
- Entry at next Monday 00:00 UTC open price
- Hold for 1 week (until next rebalance)
- If signal unchanged at rebalance, keep holding (no trade, no cost)
- If signal flips, close old position + open new (2 round-trip costs)

### Position sizing
- Fixed 1/6 of equity per coin (equal weight)
- Volatility-adjusted variant tested as parameter

### Costs
- HORROR mode: 0.20% slip + 0.12% taker = 0.64% round trip
- Note: weekly rebalance means at MOST 6 trades/week = 312/year if signals constantly flip
- In practice expect ~80-150 trades/year if signals stable (typical for TSMOM)

## Locked Parameter Search Space

| Parameter | Search values | Notes |
|-----------|---------------|-------|
| `lookback_weeks` | [1, 2, 4, 8, 12] | 1-12 weeks per Dobrynskaya finding; momentum→reversal after ~1 month so longer lookbacks may flip |
| `signal_threshold` | [0.00, 0.02, 0.05] | 0 = any direction; 2-5% creates dead zone to reduce noise |
| `vol_target` | [False, True] | Equal $ vs volatility-adjusted sizing |
| `rebalance` | weekly (locked) | hold 1 week, no parameter |
| `universe` | 6 coins (locked) | BTC ETH SOL AVAX XRP BNB |

**Total trials:** 5 × 3 × 2 = **30**

Smaller grid than v1 (1500) / v2 (300) because TSMOM has fewer levers — design choice favors interpretability over exhaustive search.

## Falsifiable Predictions

| # | Test | Threshold | Stage |
|---|------|-----------|-------|
| 1 | At least 1 of 30 IS trials has PF > 1.0 | required to proceed | 2 |
| 2 | OOS-1 PF retention | ≥ 70% of IS PF | 3 |
| 3 | Walk-forward profitable months | ≥ 80% | 4 |
| 4 | Holdout PF | > 1.3 | 5 |
| 5 | Holdout Sharpe | > 1.0 | 5 |
| 6 | Holdout MaxDD | < 30% | 5 |
| 7 | Deflated Sharpe (N=30) | > 0 | 5 |
| 8 | Multi-coin: pass on ≥ 4 of 6 coins individually | required | 6 (built-in here) |

Note: since v3 is multi-asset by design, Stage 6 robustness is partly baked into Stage 2 (we'll have per-coin numbers in the IS grid).

**Secondary prediction**: if momentum is real on crypto, longer lookbacks (8-12 weeks) should fail at signal_threshold=0 (signal flips too much), while threshold=2-5% should help — confirming the underlying behavioral story.

## Pre-Commitment Statement

> I have not examined any TSMOM backtest on our specific 6-coin universe prior to this commit. The parameter ranges are a-priori judgments based on the published academic literature on TSMOM (Moskowitz et al 2012 for equities/commodities, Han et al 2023 for crypto, Dobrynskaya 2021 for crypto momentum/reversal horizon).
>
> v3 uses entirely independent parameters from v1+v2; no failed-result is being recycled to bias v3.
>
> If v3 fails, a v4 must use a different mechanism (e.g., pairs trading, dual-asset allocation, or filter combination) — repeated TSMOM tweaking after results is forbidden.
>
> — Anthony Chung, 2026-05-24